# ML-08 — Search Visibility Modeling

Goal: predict whether a client-content item will receive a GSC click tomorrow, then compare a learned model with the transparent Week-4 rule on the same holdout rows.


## 1. Method choice and why

I use Logistic Regression because this is a binary observed outcome, it produces ranking probabilities, and its coefficients are readable. It is an honest first model for this lane; complexity is not rewarded unless it beats the rule.


## 2. Split design

Rows are daily March observations with a next-day label. I hold out entire clients using GroupShuffleSplit, so no client appears in both training and test. This tests generalization to unseen clients while keeping the same March slice and metric as the baseline.


In [8]:
from pathlib import Path
import os,duckdb,pandas as pd,numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score,average_precision_score,precision_score

p=list((Path.home()/'.cache'/'huggingface'/'hub').rglob('fact_content_daily_performance/month=2026-03/data_0.parquet'))

if p: 
    path=str(p[0])
else:
    try:
        from google.colab import userdata; 
        token=userdata.get('HF_TOKEN')
    except Exception: 
        token=os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_HUB_TOKEN')
    if not token: 
        raise RuntimeError('Set HF_TOKEN as a Colab Secret or environment variable; never paste it here.')
    from huggingface_hub import hf_hub_download; 
    path=hf_hub_download('FlyRank/internship-warehouse','fact_content_daily_performance/month=2026-03/data_0.parquet',repo_type='dataset',token=token)

safe=path.replace(chr(39),chr(39)*2); 

con=duckdb.connect(); 
con.execute(f"CREATE VIEW march AS SELECT * FROM read_parquet('{safe}')")
sql="""WITH p AS (SELECT report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,NULLIF(gsc_avg_position,0) pos,LEAD(gsc_clicks) OVER(PARTITION BY client_hash_id,content_hash_id ORDER BY report_date) nxt FROM march WHERE gsc_data_available IS TRUE) SELECT report_date,client_hash_id,content_hash_id,log(1+gsc_impressions) log_impressions,log(1+gsc_clicks) log_clicks,COALESCE(pos,100.0) avg_position,COALESCE(gsc_clicks::DOUBLE/NULLIF(gsc_impressions,0),0.0) ctr,(nxt>0)::INTEGER y FROM p WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-20' AND nxt IS NOT NULL"""
df=con.sql(sql).df(); 

features=['log_impressions','log_clicks','avg_position','ctr']; 
print('Rows:',len(df),'\nClients:',df.client_hash_id.nunique(),'\nFeatures:',features,'\nBase rate:',round(df.y.mean(),4))

gss=GroupShuffleSplit(n_splits=1,test_size=.25,random_state=202603); 
train_idx,test_idx=next(gss.split(df,df.y,groups=df.client_hash_id)); 
train=df.iloc[train_idx].copy(); 
test=df.iloc[test_idx].copy(); 
print('Train/test Shapes:',train.shape,test.shape,'\nClient overlap:',len(set(train.client_hash_id)&set(test.client_hash_id)))


Rows: 2229796 
Clients: 45 
Features: ['log_impressions', 'log_clicks', 'avg_position', 'ctr'] 
Base rate: 0.1192
Train/test Shapes: (1750817, 8) (478979, 8) 
Client overlap: 0


## 3. Train and compare with Week-4 baseline

The baseline is recomputed on these exact test rows: eligible means impressions ≥500, position 4–20, and CTR <0.5%; its score is the Week-4 transparent score. Both methods are evaluated with ROC-AUC, average precision, and precision@50.


In [9]:
model=make_pipeline(StandardScaler(),LogisticRegression(max_iter=300,random_state=202603)); 
model.fit(train[features],train.y); 

test['model_score']=model.predict_proba(test[features])[:,1]
test['baseline_score']=100*np.log1p(np.exp(test.log_impressions)-1)+20*(21-test.avg_position)-100*(100*test.ctr); 

eligible=(np.exp(test.log_impressions)-1>=500)&test.avg_position.between(4,20)&(100*test.ctr<.5); 
# test.loc[~eligible,'baseline_score']=-np.inf
test.loc[~eligible, "baseline_score"] = -1e9

def p_at_k(score,y,k=50): 
    return float(np.asarray(y)[np.argsort(-np.asarray(score))[:min(k,len(y))]].mean())

rows=[]

for name,score in [('Week-4 rule',test.baseline_score),('Logistic Regression',test.model_score)]: 
    rows.append({'method':name,'ROC_AUC':roc_auc_score(test.y,score),'average_precision':average_precision_score(test.y,score),'precision_at_50':p_at_k(score,test.y)})

comparison=pd.DataFrame(rows); 
display(comparison.round(4)); 
print('Same test rows and same metrics used for both methods.')


,method,ROC_AUC,average_precision,precision_at_50
0,Week-4 rule,0.5000,0.1386,0.08
1,Logistic Regression,0.8458,0.5224,0.94


Same test rows and same metrics used for both methods.


## 4. Errors and interpretation

False positives are pages the method ranks highly but that receive no click tomorrow; false negatives receive a click but are ranked lower. The model is intentionally evaluated as a ranking system, not as a causal effect estimator.


In [10]:
coef=pd.Series(model[-1].coef_[0],index=features).sort_values(key=abs,ascending=False); 
print('Standardized coefficient magnitude (direction shown):'); 
display(coef.to_frame('coefficient'))

test['predicted']=test.model_score>=test.model_score.quantile(.9); 
test['error_type']=np.select([test.predicted&(test.y==0),(~test.predicted)&(test.y==1)],['false_positive','false_negative'],'correct'); 
print('Top error counts:'); 

display(test.error_type.value_counts().to_frame('n'))
print('Three representative hard cases (IDs shown only as pseudonyms):'); 

display(test[test.error_type!='correct'][['client_hash_id','content_hash_id','avg_position','ctr','y','model_score','error_type']].head(3))
print('Interpretation: higher current clicks/impressions and CTR-related signals dominate because they summarize recent search demand; position is directionally useful. Hard cases are volatile day-to-day outcomes and sparse impressions, where tomorrow is inherently noisy.')


Standardized coefficient magnitude (direction shown):


,coefficient
log_impressions,1.301150
avg_position,-0.630085
log_clicks,0.400869
ctr,0.045436


Top error counts:


,n
error_type,
correct,418414
false_negative,39534
false_positive,21031


Three representative hard cases (IDs shown only as pseudonyms):


,client_hash_id,content_hash_id,avg_position,ctr,y,model_score,error_type
2856,client_23a62021009f63c4,content_01e13ce4eafc1484,1.625000,0.0,1,0.055380,false_negative
2864,client_23a62021009f63c4,content_02cff3dc5a236236,39.000000,0.0,1,0.136933,false_negative
2887,client_23a62021009f63c4,content_031977ce7879188d,11.965517,0.0,1,0.113007,false_negative


Interpretation: higher current clicks/impressions and CTR-related signals dominate because they summarize recent search demand; position is directionally useful. Hard cases are volatile day-to-day outcomes and sparse impressions, where tomorrow is inherently noisy.


## 5. Self-check

- [x] Method choice fits a binary ranking question and is interpretable.
- [x] Client-grouped split prevents client leakage.
- [x] Baseline and model use the same test rows and metrics.
- [x] Errors, coefficients, and limitations are inspected.
- [x] No future-window fields or labels are model inputs.
